In [ ]:
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/more-than-a-hack-2027'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/Thiagouh/more-than-a-hack-2027.git', REPO], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO}/requirements-starter.txt'], check=True)
os.environ['MTAH_DATA_DIR'] = '/content/drive/MyDrive/mtah-data'
os.chdir(REPO)
sys.path[:0] = [REPO, f'{REPO}/starter']
print('ok', os.getcwd())


# Vídeo com minimapa de tracking
Execute as células em ordem. O padrão gera 2 minutos (120 segundos) desde o frame 11185, sem copiar o vídeo original do Drive. O player fica disponível para sua inspeção no Colab.

O radar semitransparente no canto inferior direito mostra o campo completo com jogadores em azul/vermelho e a bola em amarelo. As posições vêm do tracking existente. A bola desaparece quando não há observação; posições ausentes não são inventadas. O radar mantém a orientação das coordenadas do ATD, sem girar com a câmera. Não há caixas nem pontos desenhados sobre jogadores no vídeo.

`MINIMAP_WIDTH` controla o tamanho e `MINIMAP_OPACITY` a opacidade (0 a 1). `SHOW_TRACK_IDS` habilita números no radar: são slots de tracking, não identidades ou camisas. `SHOW_PITCH_LINES` permite exibir as linhas do campo projetadas pela homografia sobre o vídeo, desligadas por padrão. O minimapa usa diretamente as coordenadas do campo e continua funcionando quando a homografia está indisponível.

O resultado fica em `/content/clips`, no runtime temporário, sem gravação no Drive ou envio a serviços de IA. Não compartilhe o notebook com saídas privadas salvas.


In [ ]:
# Para a partida inteira: START_FRAME = 0 e DURATION_SECONDS = None.
# Nesse caso a renderização pode levar bastante tempo e ocupar espaço no runtime.
START_FRAME = 11185
DURATION_SECONDS = 120
OUTPUT_WIDTH = 960
SHOW_TRACK_IDS = False
SHOW_PITCH_LINES = False
MINIMAP_WIDTH = 280
MINIMAP_OPACITY = 0.85
OUTPUT = '/content/clips/tracking_minimap_2min.mp4'

import shutil
if __import__('importlib').util.find_spec('cv2') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'], check=True)
if shutil.which('ffmpeg') is None:
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'], check=True)

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import games
from starter.baseline import team_of_each_track
from starter.load_homography import project, MATRIX_COLUMNS
paths = games.asset_paths(source='tactical')
for key in ('video', 'atd_meta', 'atd_raw', 'homography'):
    if not Path(paths[key]).is_file():
        raise FileNotFoundError(f'Arquivo necessário ausente: {paths[key]}')
print('Dependências e arquivos: ok')


In [ ]:
# Preparar índices sem mostrar tabelas, coordenadas ou frames.
cap = cv2.VideoCapture(paths['video'])
if not cap.isOpened():
    raise RuntimeError('Não foi possível abrir o vídeo no Drive.')
fps = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
source_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
source_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
if not np.isfinite(fps) or abs(fps - 25) > 0.01:
    raise ValueError('FPS diferente de 25; conferir alinhamento antes de renderizar.')
if not isinstance(START_FRAME, int) or not 0 <= START_FRAME < n_frames:
    raise ValueError('START_FRAME fora do vídeo.')
if DURATION_SECONDS is not None and DURATION_SECONDS <= 0:
    raise ValueError('Duração precisa ser positiva.')
end_frame = n_frames if DURATION_SECONDS is None else min(n_frames, START_FRAME + round(DURATION_SECONDS * fps))
if end_frame <= START_FRAME:
    raise ValueError('Trecho vazio.')
if OUTPUT_WIDTH < 160:
    raise ValueError('OUTPUT_WIDTH precisa ser pelo menos 160.')
out_w = int(OUTPUT_WIDTH) // 2 * 2
out_h = max(2, round(source_h * out_w / source_w / 2) * 2)
scale = np.array([out_w / source_w, out_h / source_h])
teams = team_of_each_track(paths['atd_meta'])
hom = pd.read_csv(paths['homography'])
if len(hom) != n_frames or not np.array_equal(hom['frame'].to_numpy(), np.arange(n_frames)):
    raise ValueError('Homografia e vídeo não têm índices correspondentes.')
H = hom[MATRIX_COLUMNS].to_numpy(dtype=float).reshape(-1, 3, 3)
valid = hom['valid'].to_numpy() == 1

# Geometria em coordenadas normalizadas; dimensões em metros do repositório.
length, width = games.GAMES[games.GAME]['pitch']
lines = [np.array([[0,0],[1,0],[1,1],[0,1],[0,0]], float),
         np.array([[.5,0],[.5,1]], float)]
theta = np.linspace(0, 2*np.pi, 100)
lines.append(np.column_stack((.5 + 9.15/length*np.cos(theta), .5 + 9.15/width*np.sin(theta))))
for depth, box_width in ((16.5,40.32),(5.5,18.32)):
    d, y0, y1 = depth/length, .5-box_width/width/2, .5+box_width/width/2
    lines.extend([np.array([[0,y0],[d,y0],[d,y1],[0,y1]]),
                  np.array([[1,y0],[1-d,y0],[1-d,y1],[1,y1]])])
print('Alinhamento e preparação: ok')


In [ ]:
def read_tracking(raw_path, start, stop):
    expected = start
    with open(raw_path) as handle:
        for line in handle:
            parts = line.strip().split(':')
            frame = int(parts[0]) - 1
            if frame < start:
                continue
            if frame >= stop:
                break
            if frame != expected or len(parts) != 3:
                raise ValueError(f'ATD fora de sequência ou malformado no frame {expected}.')
            players = np.array([list(map(float, item.split(','))) for item in parts[1].split(';') if item], dtype=float)
            ball = np.array(list(map(float, parts[2].rstrip(';').split(','))), dtype=float)
            if players.ndim != 2 or len(players) != len(teams) or players.shape[1] < 2 or len(ball) < 2:
                raise ValueError(f'Formato inesperado do ATD no frame {frame}.')
            yield frame, players[:, :2], ball[:2]
            expected += 1
    if expected != stop:
        raise ValueError('Tracking terminou antes do trecho solicitado.')

def pixels(matrix, points):
    with np.errstate(divide='ignore', invalid='ignore'):
        x, y = project(matrix, points[:,0], points[:,1])
    return np.column_stack((x,y)) * scale

def draw_minimap(image, players, ball):
    # Radar fixo: usa diretamente as coordenadas normalizadas do ATD.
    # A homografia é necessária somente para as linhas opcionais no vídeo.
    margin = 14
    map_w = min(int(MINIMAP_WIDTH), out_w - 2*margin)
    map_h = round(map_w * width / length)
    map_h = min(map_h, out_h - 2*margin - 48)
    if map_w < 60 or map_h < 40:
        raise ValueError('Resolução insuficiente para o minimapa.')
    x0, y0 = out_w-map_w-margin, out_h-map_h-margin
    panel = np.full((map_h,map_w,3), (35,65,35), dtype=np.uint8)
    pad = 9
    extent = np.array([map_w-2*pad-1,map_h-2*pad-1])
    def point(xy):
        return tuple(np.rint(pad + np.asarray(xy)*extent).astype(int))
    for line in lines:
        cv2.polylines(panel,[np.array([point(xy) for xy in line],np.int32)],False,(170,195,170),1,cv2.LINE_AA)
    for slot, xy in enumerate(players):
        if not np.isfinite(xy).all() or not ((xy >= 0).all() and (xy <= 1).all()):
            continue
        p = point(xy)
        color = (255,150,50) if teams[slot] == 0 else (80,80,255)
        cv2.circle(panel,p,4,(10,10,10),-1,cv2.LINE_AA)
        cv2.circle(panel,p,3,color,-1,cv2.LINE_AA)
        if SHOW_TRACK_IDS:
            cv2.putText(panel,str(slot+1),(p[0]+4,p[1]-4),cv2.FONT_HERSHEY_SIMPLEX,.28,(255,255,255),1,cv2.LINE_AA)
    if np.isfinite(ball).all() and (ball >= 0).all() and (ball <= 1).all():
        p = point(ball)
        cv2.circle(panel,p,5,(10,10,10),-1,cv2.LINE_AA)
        cv2.circle(panel,p,3,(0,255,255),-1,cv2.LINE_AA)
    else:
        cv2.putText(panel,'Bola sem tracking',(10,map_h-3),cv2.FONT_HERSHEY_SIMPLEX,.3,(220,220,220),1,cv2.LINE_AA)
    roi = image[y0:y0+map_h,x0:x0+map_w]
    cv2.addWeighted(panel,MINIMAP_OPACITY,roi,1-MINIMAP_OPACITY,0,dst=roi)
    cv2.rectangle(image,(x0,y0),(x0+map_w-1,y0+map_h-1),(180,200,180),1,cv2.LINE_AA)

def draw_overlay(image, frame, players, ball):
    usable = valid[frame] and np.isfinite(H[frame]).all()
    if usable:
        if SHOW_PITCH_LINES:
            for line in lines:
                xy = pixels(H[frame], line)
                for a,b in zip(xy[:-1], xy[1:]):
                    if np.isfinite([a,b]).all() and np.abs([a,b]).max() < 1e7:
                        ok, p, q = cv2.clipLine((0,0,out_w,out_h), tuple(np.rint(a).astype(int)), tuple(np.rint(b).astype(int)))
                        if ok:
                            cv2.line(image,p,q,(90,230,90),1,cv2.LINE_AA)
    cv2.rectangle(image,(0,0),(out_w,46),(20,20,20),-1)
    status = '' if usable else ' | homografia indisponivel'
    cv2.putText(image,f'Frame {frame} | {frame/fps:.2f}s{status}',(12,19),cv2.FONT_HERSHEY_SIMPLEX,.48,(255,255,255),1,cv2.LINE_AA)
    cv2.putText(image,'Radar: H azul | A vermelho | bola amarela | IDs opcionais: slots',(12,38),cv2.FONT_HERSHEY_SIMPLEX,.42,(255,255,255),1,cv2.LINE_AA)
    draw_minimap(image, players, ball)
    return image
print('Renderizador: ok')


In [ ]:
# Ler diretamente do Drive; somente o resultado é escrito em /content/clips.
Path(OUTPUT).parent.mkdir(parents=True, exist_ok=True)
if Path(OUTPUT).exists():
    raise FileExistsError('O resultado já existe. Altere OUTPUT para preservar o arquivo anterior.')
cap = cv2.VideoCapture(paths['video'])
encoder = None
log_path = str(Path(OUTPUT).with_suffix('.ffmpeg.log'))
try:
    if not cap.isOpened() or not cap.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME):
        raise RuntimeError('Falha ao posicionar o vídeo.')
    if abs(cap.get(cv2.CAP_PROP_POS_FRAMES) - START_FRAME) > .5:
        raise RuntimeError('Decodificador não confirmou o frame inicial.')
    with open(log_path, 'w') as log:
        encoder = subprocess.Popen(['ffmpeg','-n','-hide_banner','-loglevel','error',
            '-f','rawvideo','-pix_fmt','bgr24','-s',f'{out_w}x{out_h}',
            '-r',str(fps),'-i','pipe:0','-an','-c:v','libx264','-preset','fast',
            '-crf','25','-pix_fmt','yuv420p','-movflags','+faststart',OUTPUT],
            stdin=subprocess.PIPE, stderr=log)
        try:
            for frame, players, ball in tqdm(read_tracking(paths['atd_raw'], START_FRAME, end_frame),
                                            total=end_frame-START_FRAME, desc='Renderizando'):
                ok, image = cap.read()
                if not ok:
                    raise RuntimeError(f'Falha de leitura do vídeo no frame {frame}.')
                image = cv2.resize(image,(out_w,out_h),interpolation=cv2.INTER_AREA)
                encoder.stdin.write(draw_overlay(image,frame,players,ball).tobytes())
        finally:
            encoder.stdin.close()
            returncode = encoder.wait()
        if returncode:
            raise RuntimeError('ffmpeg falhou: ' + Path(log_path).read_text()[-3000:])
finally:
    cap.release()

# Verificação técnica, sem mostrar frames nem tracking.
probe = subprocess.run(['ffprobe','-v','error','-select_streams','v:0','-count_frames',
    '-show_entries','stream=nb_read_frames','-of','json',OUTPUT],capture_output=True,text=True,check=True)
written = int(__import__('json').loads(probe.stdout)['streams'][0]['nb_read_frames'])
if written != end_frame - START_FRAME:
    raise RuntimeError('Quantidade de frames do resultado diferente do trecho solicitado.')
print('Vídeo gerado e contagem de frames conferida: ok')


In [ ]:
from IPython.display import Video, display
# Sem autoplay. Trechos pequenos são embutidos; arquivos grandes são servidos pelo runtime.
# O modo não embutido evita carregar uma partida inteira como base64 na memória do notebook.
embed = Path(OUTPUT).stat().st_size < 40 * 1024 * 1024
display(Video(OUTPUT, embed=embed, width=min(out_w, 1100), html_attributes='controls preload="metadata"'))
